# Importing Libraries

In [66]:
import os
import numpy as np
import pandas as pd
import re
from datetime import datetime
import calendar

# Importing Dataset

In [67]:
shark_data = pd.read_excel("./Data/raw_data_GSAF5.xls", engine='xlrd')

In [68]:
shark_data.head()

,Date,Year,Type,Country,State,Location,Activity,Name,Sex,Age,...,Species,Source,pdf,href formula,href,Case Number,Case Number.1,original order,Unnamed: 21,Unnamed: 22
0,18th March,2026.0,Unprovoked,USA,California,Big River Beach Mendocino County,Surfing,Unknown,M,39,...,Great White Shark,US Sun: Mendocino Coast News:Melissa Michaelson:,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,14th March,2026.0,Unprovoked,Australia,Western Australia,Exmouth,Swimming,Unknown,F,?,...,Unknown shark 5ft (1.5m),People Magazine: Kevin McMurray Trackingsharks...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,10th March,2026.0,Unprovoked,Australia,Western Australia,Exmouth,Wing Foiling,Dave Daniell,M,?,...,Great White Shark 10ft (3m),Perth Now: Kevin McMurray Trackingsharks.com,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,5th March,2026.0,Unprovoked,Australia,Queensland,Lady Elliott Island,Snorkeling,Unknown,M,50's,...,Unknown,News.com.au: ABC News: Andy Currie,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,22nd February,2026.0,Unprovoked,New Caledonia,Noumea,Anse Vata near Point Magnin,Wing Foiling,Cyril Chevalier,M,55,...,Tiger or bull shark implicated,Johannes Marchand: Andy Currie,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Cleaning Data

In [69]:
# check size for dataframe
shark_data.shape

(7082, 23)

In [70]:
# dropping columns: 1) with PII 2) legacy columns from excel file 3) not useful for future analysis
shark_data = shark_data.drop(columns=[
    "Unnamed: 21",
    "Unnamed: 22",
    "Name",
    "Source",
    "Case Number",
    "Case Number.1",
    "original order",
    "pdf",
    "href",
    "href formula",
    "Age",
    "Species "
])

In [71]:
shark_data.head()

,Date,Year,Type,Country,State,Location,Activity,Sex,Injury,Fatal Y/N,Time
0,18th March,2026.0,Unprovoked,USA,California,Big River Beach Mendocino County,Surfing,M,Injuries to both legs and feet,N,1715hrs
1,14th March,2026.0,Unprovoked,Australia,Western Australia,Exmouth,Swimming,F,Minor injuries,N,1015hrs
2,10th March,2026.0,Unprovoked,Australia,Western Australia,Exmouth,Wing Foiling,M,None reported damage to board,N,0800hrs
3,5th March,2026.0,Unprovoked,Australia,Queensland,Lady Elliott Island,Snorkeling,M,Lacerations elbow and abdomen,N,0815hrs
4,22nd February,2026.0,Unprovoked,New Caledonia,Noumea,Anse Vata near Point Magnin,Wing Foiling,M,Deep wounds arms and legs,Y,?


In [72]:
# Lowercasing columns names
shark_data.columns = shark_data.columns.str.lower()

In [73]:
shark_data.head()

,date,year,type,country,state,location,activity,sex,injury,fatal y/n,time
0,18th March,2026.0,Unprovoked,USA,California,Big River Beach Mendocino County,Surfing,M,Injuries to both legs and feet,N,1715hrs
1,14th March,2026.0,Unprovoked,Australia,Western Australia,Exmouth,Swimming,F,Minor injuries,N,1015hrs
2,10th March,2026.0,Unprovoked,Australia,Western Australia,Exmouth,Wing Foiling,M,None reported damage to board,N,0800hrs
3,5th March,2026.0,Unprovoked,Australia,Queensland,Lady Elliott Island,Snorkeling,M,Lacerations elbow and abdomen,N,0815hrs
4,22nd February,2026.0,Unprovoked,New Caledonia,Noumea,Anse Vata near Point Magnin,Wing Foiling,M,Deep wounds arms and legs,Y,?


In [74]:
# renaming columns
shark_data = shark_data.rename(columns = {'fatal y/n' : 'fatal_yes_no',
                             'type' : 'attack_type',
                             'sex' : 'gender'
                                         })

In [75]:
shark_data.head()

,date,year,attack_type,country,state,location,activity,gender,injury,fatal_yes_no,time
0,18th March,2026.0,Unprovoked,USA,California,Big River Beach Mendocino County,Surfing,M,Injuries to both legs and feet,N,1715hrs
1,14th March,2026.0,Unprovoked,Australia,Western Australia,Exmouth,Swimming,F,Minor injuries,N,1015hrs
2,10th March,2026.0,Unprovoked,Australia,Western Australia,Exmouth,Wing Foiling,M,None reported damage to board,N,0800hrs
3,5th March,2026.0,Unprovoked,Australia,Queensland,Lady Elliott Island,Snorkeling,M,Lacerations elbow and abdomen,N,0815hrs
4,22nd February,2026.0,Unprovoked,New Caledonia,Noumea,Anse Vata near Point Magnin,Wing Foiling,M,Deep wounds arms and legs,Y,?


In [76]:
# checking for NaN values
shark_data.isnull().sum() 

date               0
year               2
attack_type       18
country           50
state            487
location         567
activity         583
gender           578
injury            35
fatal_yes_no     561
time            3527
dtype: int64

In [77]:
# checking unique values in all columns to check which columns need further cleaning
shark_data.nunique()

date            6121
year             261
attack_type       13
country          253
state            946
location        4626
activity        1612
gender            11
injury          4193
fatal_yes_no      12
time             476
dtype: int64

### Standardizing certain columns - too manu unique values and different types

In [131]:
# standardizing data in column activity - type to sring, stripping and lowercasing
shark_data['activity'] = shark_data['activity'].where(shark_data['activity'].isna(), shark_data['activity'].astype(str)).str.strip().str.lower()

In [79]:
# standardizing data in column attack_type - stripping and lowercasing
shark_data["attack_type"] = shark_data["attack_type"].str.lower().str.strip()

In [80]:
# checking the attack_type for unique values 
shark_data['attack_type'].value_counts()

attack_type
unprovoked             5234
provoked                644
invalid                 552
watercraft              355
sea disaster            242
questionable             26
boat                      7
?                         1
unconfirmed               1
unverified                1
under investigation       1
Name: count, dtype: int64

In [81]:
# defining function and calling to standardize

def attack_type_cleaning(attack):
    if pd.isna(attack):
        return np.nan
    result = attack.strip().lower()
    if result in ['unprovoked']:
        return 'Unprovoked'
    elif result in ['provoked']:
        return 'Provoked'
    elif result in ['invalid', 'questionable', 'unconfirmed', 'unverified', '?', 'under investigation']:
        return np.nan
    elif result in ['watercraft', 'boat', 'sea disaster']:
        return 'Watercraft'
    else:
        return np.nan

shark_data['attack_type'] = shark_data['attack_type'].apply(attack_type_cleaning)  

In [82]:
# checking if changes were saved and correct
shark_data['attack_type'].value_counts()

attack_type
Unprovoked    5234
Provoked       644
Watercraft     604
Name: count, dtype: int64

In [83]:
# checking if changes were saved and correct
shark_data['attack_type'].isnull().sum()

np.int64(600)

In [84]:
# checking fatal_yes_no for unique values
shark_data['fatal_yes_no'].value_counts()

fatal_yes_no
N          4938
Y          1491
UNKNOWN      71
 N            7
F             5
M             3
n             1
Nq            1
2017          1
Y x 2         1
N             1
y             1
Name: count, dtype: int64

In [85]:
# standardizing data in column fatal_yes_no - stripping and lowercasing
shark_data['fatal_yes_no'] = shark_data['fatal_yes_no'].str.strip().str.lower()

In [86]:
# checking if changes were saved and correct
shark_data['fatal_yes_no'].value_counts()

fatal_yes_no
n          4947
y          1492
unknown      71
f             5
m             3
nq            1
y x 2         1
Name: count, dtype: int64

In [87]:
# defining function and calling to standardize

def fatal_cleaning(result):
    if pd.isna(result):
        return np.nan
    result = result.strip()
    if result in ['n', 'nq']:
        return 'No'
    elif result in ['y', 'y x 2']:
        return 'Yes'
    else:
        return np.nan

shark_data['fatal_yes_no'] = shark_data['fatal_yes_no'].apply(fatal_cleaning)      

In [88]:
# checking if changes were saved and correct
shark_data['fatal_yes_no'].value_counts()

fatal_yes_no
No     4948
Yes    1493
Name: count, dtype: int64

In [89]:
# checking if changes were saved and correct
shark_data['fatal_yes_no'].isnull().sum()

np.int64(641)

In [90]:
# standardizing data in column gender - stripping and lowercasing
shark_data['gender'] = shark_data['gender'].str.strip().str.lower()

In [91]:
# checking gender column for unique values
shark_data['gender'].value_counts()

gender
m        5685
f         812
?           2
n           2
lli         1
m x 2       1
.           1
Name: count, dtype: int64

In [92]:
# defining function and calling to standardize

def gender_cleaning(gender):
    if pd.isna(gender):
        return np.nan
    if gender in ['?', 'n', 'lli', 'm x 2', '.']:
        return np.nan
    else:
        return gender

shark_data['gender'] = shark_data['gender'].apply(gender_cleaning)  

In [93]:
# uppercasing the gender
shark_data['gender'] = shark_data['gender'].str.upper()

In [94]:
# checking if changes were saved and correct
shark_data['gender'].value_counts()

gender
M    5685
F     812
Name: count, dtype: int64

In [95]:
# checking if changes were saved and correct
shark_data['gender'].isnull().sum()

np.int64(585)

In [96]:
# checking year column for unique values
shark_data['year'].value_counts()

year
2015.0    143
2017.0    141
2016.0    133
0.0       129
2011.0    128
         ... 
1784.0      1
1783.0      1
1780.0      1
1779.0      1
1816.0      1
Name: count, Length: 261, dtype: int64

In [97]:
# dropping data from years 1900 and before (NaN included) and setting data type as integer
shark_data = shark_data[shark_data["year"].notna()]
shark_data = shark_data[shark_data["year"] >= 1900]
shark_data["year"] = shark_data["year"].astype(int)

In [98]:
# checking if changes were saved and correct
shark_data['year'].value_counts()

year
2015    143
2017    141
2016    133
2011    128
2014    126
       ... 
1921     12
1917     12
1903     10
1901      9
1918      5
Name: count, Length: 127, dtype: int64

In [99]:
# checking if changes were saved and correct
shark_data['year'].isnull().sum()

np.int64(0)

In [100]:
# standardizing data in column country - stripping and lowercasing
shark_data['country'] = shark_data['country'].str.lower().str.strip()

In [101]:
# dropping rows in which we have no way of identifying country (country, state and location null)
shark_data.dropna(subset=['country', 'state', 'location'], how = 'all', inplace=True)

In [102]:
# checking country column for null values
shark_data['country'].isnull().sum()

np.int64(12)

In [103]:
# checking info in null values row to take decision on keeping or dropping
shark_data[shark_data['country'].isnull()]

,date,year,attack_type,country,state,location,activity,gender,injury,fatal_yes_no,time
3711,Ca. 1983,1983,Unprovoked,NaN,English Channel,NaN,swimming,M,Left leg bitten,No,NaN
4150,02-Aug-1970,1970,NaN,NaN,Caribbean Sea,Between St. Kitts & Nevis,sea disaster sinking of ferryboat christina,NaN,"Sharks scavenged on bodies, but no record of t...",NaN,Afternoon
4368,21-Oct-1965,1965,Unprovoked,NaN,NaN,Florida Strait,the boat caribou ii sank,M,Survived,No,NaN
4781,26-Jan-1960,1960,Watercraft,NaN,"Between Timor & Darwin, Australia",NaN,portuguese airliner with 9 people aboard went ...,NaN,"As searchers approached wreckage, sharks circl...",No,NaN
4994,13-Sep-1956,1956,Unprovoked,NaN,Near the Andaman & Nicobar Islands,NaN,climbing back on ship,M,FATAL,Yes,P.M.
5029,1956,1956,Watercraft,NaN,Between Comores & Madagascar,Geyser Bank,shipwreck,M,FATAL,Yes,NaN
5264,Dec-1949,1949,Watercraft,NaN,Caribbean Sea,Between Cuba & Costa Rica,"sea disaster, sinking of the motorship wingate",M,Fatal or drowning or scavenging,Yes,NaN
5405,24-Oct-1944,1944,Watercraft,NaN,NaN,225 miles east of Hong Kong,japanese pow ship arisan maru with 1800 americ...,M,Most of the men drowned & some were taken by s...,Yes,>17h30
5466,Nov-1942,1942,Watercraft,NaN,Off South American coast,NaN,dutch merchant ship zaandam torpedoed by the ...,M,FATAL,Yes,NaN
5478,Jun-1942,1942,Unprovoked,NaN,300 miles east of St. Thomas (Virgin Islands),NaN,on life raft tethered to lifeboat. a seaman pu...,M,Forearm lacerated,No,NaN


In [104]:
# dropping null values in country, as they are not related to active behaviour of people during travelling
shark_data.dropna(subset=['country'], inplace=True)

In [105]:
# checking if dropping worked
shark_data['country'].isnull().sum()

np.int64(0)

In [106]:
# checking unique values for column country 
# done as list to be able to see all data at once
print(shark_data['country'].unique().tolist())

['usa', 'australia', 'new caledonia', 'cayman islands', 'brazil', 'us virgin islands', 'mozambique', 'french polynesia', 'samoa', 'columbia', 'costa rica', 'bahamas', 'puerto rico', 'spain', 'canary islands', 'south africa', 'vanuatu', 'jamaica', 'israel', 'mexico', 'maldives', 'philippines', 'turks and caicos', 'egypt', 'thailand', 'new zealand', 'hawaii', 'honduras', 'indonesia', 'morocco', 'belize', 'maldive islands', 'tobago', 'india', 'trinidad', 'coral sea', 'portugal', 'colombia', 'ecuador', 'cuba', 'seychelles', 'argentina', 'fiji', 'england', 'japan', 'british overseas territory', 'canada', 'jordan', 'st kitts / nevis', 'st martin', 'papua new guinea', 'reunion island', 'china', 'ireland', 'italy', 'malaysia', 'libya', 'mauritius', 'solomon islands', 'st helena, british overseas territory', 'comoros', 'reunion', 'united kingdom', 'united arab emirates', 'cape verde', 'dominican republic', 'aruba', 'atlantic ocean', 'greece', 'st. martin', 'france', 'trinidad & tobago', 'kiriba

In [107]:
# standardizing country names, correcting spelling inconsistencies and merging duplicates
country_standardise = {
    'reunion island': 'reunion',
    'columbia': 'colombia',
    'turks & caicos': 'turks and caicos',
    'united arab emirates (uae)': 'united arab emirates',
    'st. martin': 'st martin',
    'st. maartin': 'st martin',
    'trinidad': 'trinidad and tobago',
    'tobago': 'trinidad and tobago',
    'trinidad & tobago': 'trinidad and tobago',
    'federated states of micronesia': 'micronesia',
    'western samoa': 'samoa',
    'sudan?': 'sudan',
    'maldive islands': 'maldives',
    'st kitts / nevis': 'st kitts and nevis',
    'nevis': 'st kitts and nevis',
    'okinawa': 'japan'
}

shark_data['country'] = shark_data['country'].replace(country_standardise)

# Deriving New Columns

In [108]:
# checking unique values for column country post-standardisation
print(shark_data['country'].unique().tolist())

['usa', 'australia', 'new caledonia', 'cayman islands', 'brazil', 'us virgin islands', 'mozambique', 'french polynesia', 'samoa', 'colombia', 'costa rica', 'bahamas', 'puerto rico', 'spain', 'canary islands', 'south africa', 'vanuatu', 'jamaica', 'israel', 'mexico', 'maldives', 'philippines', 'turks and caicos', 'egypt', 'thailand', 'new zealand', 'hawaii', 'honduras', 'indonesia', 'morocco', 'belize', 'trinidad and tobago', 'india', 'coral sea', 'portugal', 'ecuador', 'cuba', 'seychelles', 'argentina', 'fiji', 'england', 'japan', 'british overseas territory', 'canada', 'jordan', 'st kitts and nevis', 'st martin', 'papua new guinea', 'reunion', 'china', 'ireland', 'italy', 'malaysia', 'libya', 'mauritius', 'solomon islands', 'st helena, british overseas territory', 'comoros', 'united kingdom', 'united arab emirates', 'cape verde', 'dominican republic', 'aruba', 'atlantic ocean', 'greece', 'france', 'kiribati', 'diego garcia', 'taiwan', 'palestinian territories', 'guam', 'nigeria', 'ton

In [109]:
# checking counts of countries with '/', indicator of deviding location to take decision in keeping/dropping information
shark_data[shark_data['country'].str.contains('/', na=False)]['country'].value_counts()

country
egypt / israel                1
red sea / indian ocean        1
andaman / nicobar islandas    1
iran / iraq                   1
solomon islands / vanuatu     1
italy / croatia               1
Name: count, dtype: int64

In [110]:
# defining list of countries per hemisphere

southern = [
    'australia', 'brazil', 'mozambique', 'french polynesia', 'samoa',
    'south africa', 'vanuatu', 'new zealand', 'indonesia', 'ecuador',
    'argentina', 'fiji', 'papua new guinea', 'reunion', 'solomon islands',
    'comoros', 'mauritius', 'madagascar', 'chile', 'kenya', 'tanzania',
    'uruguay', 'american samoa', 'namibia', 'angola', 'falkland islands',
    'mayotte', 'gabon', 'tonga', 'kiribati', 'new britain', 'new guinea',
    'seychelles', 'diego garcia', 'colombia', 'java',
    'south pacific ocean', 'south atlantic ocean', 'southwest pacific ocean',
    'admiralty islands', 'st helena, british overseas territory',
]

northern = [
    'usa', 'canada', 'mexico', 'bahamas', 'cuba', 'jamaica', 'puerto rico',
    'costa rica', 'honduras', 'belize', 'spain', 'portugal', 'france',
    'england', 'ireland', 'italy', 'scotland', 'united kingdom', 'greece',
    'croatia', 'norway', 'russia', 'japan', 'china', 'india', 'egypt',
    'morocco', 'israel', 'jordan', 'saudi arabia', 'iran', 'iraq',
    'nigeria', 'senegal', 'somalia', 'sierra leone', 'liberia', 'south korea',
    'taiwan', 'vietnam', 'malaysia', 'philippines', 'thailand', 'maldives',
    'sri lanka', 'turkey', 'tunisia', 'libya', 'bermuda', 'hawaii',
    'canary islands', 'azores', 'cayman islands', 'turks and caicos',
    'trinidad and tobago', 'aruba', 'guam', 'northern mariana islands',
    'palau', 'micronesia', 'marshall islands', 'el salvador', 'panama',
    'venezuela', 'guatemala', 'nicaragua', 'haiti', 'barbados', 'antigua',
    'grenada', 'guyana', 'martinique', 'iceland', 'bangladesh', 'singapore',
    'hong kong', 'united arab emirates', 'kuwait', 'yemen', 'sudan',
    'cape verde', 'dominican republic', 'british virgin islands',
    'us virgin islands', 'st kitts and nevis', 'st martin', 'new caledonia',
    'johnston island', 'grand cayman', 'british overseas territory',
    'british isles', 'british west indies', 'netherlands antilles',
    'the balkans', 'san domingo', 'caribbean sea', 'atlantic ocean',
    'north atlantic ocean', 'north pacific ocean', 'red sea',
    'persian gulf', 'gulf of aden', 'northern arabian sea',
    'south china sea', 'bay of bengal', 'north sea', 'mid atlantic ocean',
    'mid-pacifc ocean', 'central pacific', 'coral sea', 'monaco',
    'malta', 'montenegro', 'slovenia', 'curacao', 'crete', 'cyprus',
    'palestinian territories', 'myanmar', 'scotland', 'azerbaijan',
    'north africa', 'northern ireland',
]

In [111]:
# deriving new column called hemesphere based on country column
# defining function to get hemisphere and calling it

def get_hemisphere(country):
    if pd.isna(country):
        return np.nan
    country = str(country).lower().strip()
    if country in southern:
        return 'Southern'
    if country in northern:
        return 'Northern'
    return np.nan


shark_data['hemisphere'] = shark_data['country'].apply(get_hemisphere)

In [112]:
# checking hemisphere column values to look unique values
shark_data['hemisphere'].value_counts()

hemisphere
Northern    3573
Southern    2707
Name: count, dtype: int64

In [113]:
# checking hemisphere colum for null values
shark_data['hemisphere'].isnull().sum()

np.int64(26)

In [114]:
# deriving new column called month based on date column
# defining function to get month and calling it

def only_month(date): 
    if pd.isna(date):
        return np.nan
    if isinstance(date, datetime):
        return date.month
    elif '-' not in str(date) and not re.search(r'\d{4}', str(date)):
        cleaned = str(date).strip()
        cleaned = re.sub(r'(\d)(st|nd|rd|th)', r'\1', cleaned)
        parsed = pd.to_datetime(cleaned + ' 2000', format='%d %B %Y', errors='coerce')
        return parsed.month if pd.notna(parsed) else np.nan
    else:
        cleaned = str(date).strip()
        cleaned = re.sub(r'^Reported\s+', '', cleaned)
        cleaned = cleaned.replace('- ', '-').replace(' -', '-')
        cleaned = cleaned.replace('--', '-')
        cleaned = cleaned.replace('`', '')
        parsed = pd.to_datetime(cleaned, errors='coerce')
        return parsed.month if pd.notna(parsed) else np.nan

shark_data['month'] = shark_data['date'].apply(only_month)

In [115]:
# checking month column values to look for possible discrepancies
shark_data['month'].value_counts()

month
1.0     753
7.0     717
8.0     607
9.0     559
6.0     507
10.0    472
4.0     459
12.0    455
3.0     425
11.0    422
5.0     416
2.0     394
Name: count, dtype: int64

In [116]:
# checking month colum for null values
shark_data['month'].isnull().sum()

np.int64(120)

In [117]:
# converting month column to month names for better redability
# defining function and calling it

def convert_months(month): 
    if pd.notna(month):
        return calendar.month_name[int(month)] 
    else: 
        return np.nan

shark_data['month'] = shark_data['month'].apply(convert_months)

In [118]:
# checking month column values to look for possible discrepancies
shark_data['month'].value_counts()

month
January      753
July         717
August       607
September    559
June         507
October      472
April        459
December     455
March        425
November     422
May          416
February     394
Name: count, dtype: int64

In [119]:
# creating dictionary to be able to use in function to create new time_of_day column

word_map = {
    'morning': 'Morning', 'early morning': 'Morning', 'dawn': 'Morning', 'daybreak': 'Morning',
    'early  morning': 'Morning', 'late morning': 'Morning', 'mid morning': 'Morning',
    'mid-morning': 'Morning', 'just before dawn': 'Night', 'before daybreak': 'Night',
    'a.m.': 'Morning', 'am': 'Morning',
    'afternoon': 'Afternoon', 'midday': 'Afternoon', 'noon': 'Afternoon', 'lunchtime': 'Afternoon',
    'late afternoon': 'Afternoon', 'early afternoon': 'Afternoon', 'mid afternoon': 'Afternoon',
    'late afternon': 'Afternoon', 'late afternoon': 'Afternoon', 'after noon': 'Afternoon',
    'just before noon': 'Afternoon', '"midday"': 'Afternoon', 'midday.': 'Afternoon',
    'daytime': 'Afternoon', '"after lunch"': 'Afternoon',
    'p.m.': 'Afternoon', 'pm': 'Afternoon',
    'evening': 'Evening', 'dusk': 'Evening', 'sunset': 'Evening', 'nightfall': 'Evening',
    'after dusk': 'Evening', 'after dark': 'Evening', '"after dark"': 'Evening',
    '"evening"': 'Evening', '"early evening"': 'Evening', 'just before sundown': 'Evening',
    '2 hrs before sunset': 'Evening',
    'night': 'Night', 'midnight': 'Night', 'after dark': 'Night', 'dark': 'Night',
    '"night"': 'Night', 'late night': 'Night', 'after midnight': 'Night',
    'shortly after midnight': 'Night',
}

In [120]:
# creating column time of day based on column time 
# defining function and calling it
# Night: 00h00 - 05h59 | Morning: 06h00 - 11h59 | Afternoon: 12h00 - 17h59 | Evening: 18h00 - 23h59

def tod(time):        
    if pd.isna(time):           
        return np.nan
        
    cleaned = str(time).lower().strip()
    
    if cleaned in word_map:
        return word_map[cleaned]

    cleaned_num = cleaned.replace('j', 'h').replace('hoo', 'h00').replace('`', '').replace('"', '')
    cleaned_num = re.sub(r'(\d+):(\d+)\s*pm', lambda m: f'{int(m.group(1))+12:02d}h{m.group(2)}', cleaned_num)
    cleaned_num = re.sub(r'(\d+):(\d+)\s*am', lambda m: f'{int(m.group(1)):02d}h{m.group(2)}', cleaned_num)
    cleaned_num = re.sub(r'^(\d)h', lambda m: f'0{m.group(1)}h', cleaned_num)
    cleaned_num = re.sub(r'^(\d{3})$', lambda m: f'0{m.group(1)}', cleaned_num)
    match = re.search(r'(\d{2})h\d{2}', cleaned_num) or re.search(r'(\d{2})\d{2}', cleaned_num)
    
    if match:
        hour = int(match.group(1))
        if 6 <= hour <= 11:
            return 'Morning'
        elif 12 <= hour <= 17:
            return 'Afternoon'
        elif 18 <= hour <= 23:
            return 'Evening'
        else:
            return 'Night'
    return np.nan

shark_data['time_of_day'] = shark_data['time'].apply(tod)

In [121]:
# checking month column values to look for possible discrepancies
shark_data['time_of_day'].value_counts()

time_of_day
Afternoon    1836
Morning      1121
Evening       324
Night         106
Name: count, dtype: int64

In [122]:
shark_data['time_of_day'].isnull().sum()

np.int64(2919)

In [123]:
# defining dictionaries of seasons by month and hemisphere

northern_seasons = {
    'January': 'Winter', 'February': 'Winter', 'March': 'Winter',
    'April': 'Spring', 'May': 'Spring', 'June': 'Spring',
    'July': 'Summer', 'August': 'Summer', 'September': 'Summer',
    'October': 'Autumn', 'November': 'Autumn', 'December': 'Autumn'
}

southern_seasons = {
    'January': 'Summer', 'February': 'Summer', 'March': 'Summer',
    'April': 'Autumn', 'May': 'Autumn', 'June': 'Autumn',
    'July': 'Winter', 'August': 'Winter', 'September': 'Winter',
    'October': 'Spring', 'November': 'Spring', 'December': 'Spring'
}

In [124]:
# defining function and calling function

def get_season(row):
    if pd.isna(row['month']) or pd.isna(row['hemisphere']):
        return np.nan
    if row['hemisphere'] == 'Northern':
        return northern_seasons.get(row['month'], np.nan)
    if row['hemisphere'] == 'Southern':
        return southern_seasons.get(row['month'], np.nan)
    else:
        return np.nan

shark_data['season'] = shark_data.apply(get_season, axis=1)

In [125]:
# checking season column values to look for possible discrepancies
shark_data['season'].value_counts()

season
Summer    2494
Spring    1603
Autumn    1115
Winter     948
Name: count, dtype: int64

In [126]:
# checking season colum for null values
shark_data['season'].isnull().sum()

np.int64(146)

# Checking % of null values in dataframe and shape of new dataframe before exporting

In [127]:
# checking % of missing data
shark_data.isna().sum() / len(shark_data) *100

date             0.000000
year             0.000000
attack_type      8.277831
country          0.000000
state            5.233111
location         6.121154
activity         7.373930
gender           8.214399
injury           0.396448
fatal_yes_no     8.626705
time            45.306058
hemisphere       0.412306
month            1.902950
time_of_day     46.289248
season           2.315255
dtype: float64

In [128]:
shark_data.shape

(6306, 15)

In [129]:
shark_data.head()

,date,year,attack_type,country,state,location,activity,gender,injury,fatal_yes_no,time,hemisphere,month,time_of_day,season
0,18th March,2026,Unprovoked,usa,California,Big River Beach Mendocino County,surfing,M,Injuries to both legs and feet,No,1715hrs,Northern,March,Afternoon,Winter
1,14th March,2026,Unprovoked,australia,Western Australia,Exmouth,swimming,F,Minor injuries,No,1015hrs,Southern,March,Morning,Summer
2,10th March,2026,Unprovoked,australia,Western Australia,Exmouth,wing foiling,M,None reported damage to board,No,0800hrs,Southern,March,Morning,Summer
3,5th March,2026,Unprovoked,australia,Queensland,Lady Elliott Island,snorkeling,M,Lacerations elbow and abdomen,No,0815hrs,Southern,March,Morning,Summer
4,22nd February,2026,Unprovoked,new caledonia,Noumea,Anse Vata near Point Magnin,wing foiling,M,Deep wounds arms and legs,Yes,?,Northern,February,NaN,Winter


In [130]:
# exporting cleaned dataset as .pkl
shark_data.to_pickle('./Data/shark_data_clean.pkl')